In [ ]:
import numpy as np

from utils.data import DataManager
from utils.tools.config import (
    RISK_ANALYSIS,
    ANALYSIS_START_DATE,
    ANALYSIS_END_DATE
)

from utils.analysis.risk_metrics import (
    VarEsAnalyzer, VarEsReporter,
    RatioAnalyzer, RatioReporter,
    DistributionAnalyzer, DistributionReporter,
    BenchmarkAnalyzer, BenchmarkReporter,
    DrawdownAnalyzer, DrawdownReporter,
    CorrelationAnalyzer, CorrelationReporter
)

from utils.visualizations import (
    DrawdownVisualizer, 
    DistributionVisualizer, 
    VarEsVisualizer, 
    RatioVisualizer, 
    BenchmarkVisualizer
)

from utils.analysis.risk_metrics.components import calculate_portfolio_returns

In [ ]:
# ANALYSIS CONFIGURATION
# Dates come from config.py (ANALYSIS_DATES)
# Customize here only if you need different values

# Portfolio to analyze
TICKERS  = ["SYF", "CF", "AFL", "NEM"]
BENCHMARK_NAME = "SP500"

# (Optional) Custom dates - Leave empty to use config.py
USE_CUSTOM_DATES = False
START_DATE = ""  # e.g.: "2020-01-01"
END_DATE = ""    # e.g.: "2024-12-31"

# Portfolio weights (equal weight by default)
WEIGHTS = np.ones(len(TICKERS)) / len(TICKERS)

# Constants from config
RISK_FREE_RATE = RISK_ANALYSIS['risk_free_rate']
ANNUAL_FACTOR = RISK_ANALYSIS['annual_factor']
DEFAULT_CONFIDENCE = RISK_ANALYSIS['default_confidence_level']
CONFIDENCE_LEVELS = RISK_ANALYSIS['default_confidence_levels']
ROLLING_WINDOW = RISK_ANALYSIS['rolling']['default_window']
MC_SIMULATIONS = RISK_ANALYSIS['monte_carlo']['n_simulations']
MC_SEED = RISK_ANALYSIS['monte_carlo']['seed']

# Resolve dates
if USE_CUSTOM_DATES and START_DATE and END_DATE:
    final_start, final_end = START_DATE, END_DATE
    print(f"Using custom dates: {final_start} -> {final_end}")
else:
    final_start, final_end = ANALYSIS_START_DATE, ANALYSIS_END_DATE
    print(f"Using config.py dates: {final_start} -> {final_end}")

print(f"\nPortfolio: {len(TICKERS)} assets")
print(f"Benchmark: {BENCHMARK_NAME}")
print(f"Weights: Equal weight ({1/len(TICKERS):.1%} each)")

In [ ]:
# DATA DOWNLOAD
data_manager = DataManager()

print("Downloading data...")
assets_prices, benchmark_prices = data_manager.download_portfolio_with_benchmark(
    tickers=TICKERS,
    benchmark_name=BENCHMARK_NAME,
    start_date=final_start,
    end_date=final_end
)

# Calculate returns
returns = assets_prices.pct_change().dropna()
benchmark_returns = benchmark_prices.pct_change().dropna()

print(f"\n[OK] Data downloaded:")
print(f"   Period: {assets_prices.index[0].date()} -> {assets_prices.index[-1].date()}")
print(f"   Days: {len(assets_prices)}")
print(f"   Assets: {list(returns.columns)}")

In [ ]:
# ANALYZER INITIALIZATION

# Analyzers
var_es_analyzer = VarEsAnalyzer(annual_factor=ANNUAL_FACTOR)
ratio_analyzer = RatioAnalyzer(annual_factor=ANNUAL_FACTOR)
dist_analyzer = DistributionAnalyzer()
benchmark_analyzer = BenchmarkAnalyzer(annual_factor=ANNUAL_FACTOR)
drawdown_analyzer = DrawdownAnalyzer(annual_factor=ANNUAL_FACTOR)
correlation_analyzer = CorrelationAnalyzer()

# Reporters
var_es_reporter = VarEsReporter(var_es_analyzer)
ratio_reporter = RatioReporter(ratio_analyzer)
dist_reporter = DistributionReporter(dist_analyzer)
benchmark_reporter = BenchmarkReporter(benchmark_analyzer)
drawdown_reporter = DrawdownReporter(drawdown_analyzer)
correlation_reporter = CorrelationReporter(correlation_analyzer)

# Visualizers 
drawdown_viz = DrawdownVisualizer(annual_factor=ANNUAL_FACTOR)
dist_viz = DistributionVisualizer()  
var_es_viz = VarEsVisualizer(annual_factor=ANNUAL_FACTOR)
ratio_viz = RatioVisualizer(annual_factor=ANNUAL_FACTOR)
benchmark_viz = BenchmarkVisualizer(annual_factor=ANNUAL_FACTOR)

In [ ]:
# VAR AND ES ANALYSIS
portfolio_returns = calculate_portfolio_returns(returns, WEIGHTS)

print("Calculating VaR and Expected Shortfall...\n")

var_es_result = var_es_analyzer.calculate_multi_level( 
    returns=returns,  
    weights=WEIGHTS,
    confidence_levels=CONFIDENCE_LEVELS,
    n_simulations=MC_SIMULATIONS,
    seed=MC_SEED
)

# Mostrar resultados
var_es_reporter.generate_report(
    returns=returns,
    weights=WEIGHTS,
    confidence_level=DEFAULT_CONFIDENCE,
    n_simulations=MC_SIMULATIONS,
    seed=MC_SEED
)

In [ ]:
# PERFORMANCE RATIO ANALYSIS
print("Calculating performance ratios...\n")

ratios = ratio_analyzer.calculate_all_ratios(
    returns=returns,  
    weights=WEIGHTS,
    risk_free_rate=RISK_FREE_RATE
)

ratio_reporter.generate_report(
    returns=returns,
    weights=WEIGHTS,
    risk_free_rate=RISK_FREE_RATE
)

In [ ]:
# DRAWDOWN ANALYSIS
print("Analyzing drawdowns...\n")

drawdown_result = drawdown_analyzer.analyze(
    returns=returns,  
    weights=WEIGHTS,
    risk_free_rate=RISK_FREE_RATE  
)

drawdown_reporter.generate_report(
    returns=returns,
    weights=WEIGHTS,
    risk_free_rate=RISK_FREE_RATE
)

In [ ]:
# DISTRIBUTION ANALYSIS
print("Analyzing return distribution...\n")

dist_result = dist_analyzer.analyze(
    returns=returns, 
    weights=WEIGHTS
)

dist_reporter.generate_report(
    returns=returns,
    weights=WEIGHTS
)

In [ ]:
# BENCHMARK COMPARISON ANALYSIS
print("Comparing with benchmark...\n")

benchmark_result = benchmark_analyzer.analyze(
    returns=returns,
    weights=WEIGHTS,
    benchmark_returns=benchmark_returns,
    risk_free_rate=RISK_FREE_RATE
)

benchmark_reporter.generate_report(
    returns=returns,
    weights=WEIGHTS,
    benchmark_returns=benchmark_returns,
    risk_free_rate=RISK_FREE_RATE
)

In [ ]:
# CORRELATION ANALYSIS
print("Analyzing correlations...\n")

corr_result = correlation_analyzer.analyze(returns=returns)

correlation_reporter.generate_report(returns=returns)